# Spoof Buster Lite reproduction: Colab runner

Runtime > Change runtime type > **T4 GPU**. Run the cells top to bottom. The full run takes about 1.5 to 2 hours (most of it minutiae extraction and training).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1. Get the code (replace with your public repo URL)
!git clone https://github.com/YOUR_USERNAME/spoofbuster-lite-repro.git
%cd spoofbuster-lite-repro

In [ ]:
# 2. Get the data. Copy it to local disk first: reading thousands of images from Drive is very slow.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/data
# EDIT: path to your dataset archive on Drive
!unzip -q '/content/drive/MyDrive/FPAD/dataset.zip' -d /content/raw
!find /content/raw -maxdepth 3 -type d | head -40

**3. Arrange the images** as `/content/data/bonafide/...` and `/content/data/spoof/<material>/...` (one folder per PA material). Adapt the cell below to the archive's layout, then check the counts. Write down anything you had to change: it goes in the write-up.

In [ ]:
# EDIT to match your archive, e.g.:
# !mv /content/raw/Live /content/data/bonafide
# !mkdir -p /content/data/spoof && mv /content/raw/Fake/* /content/data/spoof/
!for d in /content/data/bonafide /content/data/spoof/*; do echo "$d: $(find "$d" -type f | wc -l)"; done

In [ ]:
# 4. Pin the versions this runtime actually uses (commit environment.yml afterwards)
!python scripts/pin_env.py

In [ ]:
# 5. Everything: split, minutiae, training (2 seeds), evaluation, timing, tables and figures
%env DATA_DIR=/content/data
!bash run_all.sh 2>&1 | grep -v -E 'TensorSpec|WARNING|I0000|E0000'

In [ ]:
!cat results/headline.json
!cat results/table_main.tex

In [ ]:
# 6. Save results back to the repo. Option A: push with a GitHub personal access token
import getpass, subprocess
token = getpass.getpass('GitHub token: ')
!git config user.name 'YOUR NAME' && git config user.email 'YOUR_EMAIL'
!git add results checkpoints figures environment.yml
!git commit -m 'Results from Colab run'
subprocess.run(f'git push https://{token}@github.com/YOUR_USERNAME/spoofbuster-lite-repro.git HEAD:main', shell=True, check=True)

In [ ]:
# Option B: download and commit from your laptop
!zip -qr /content/results_bundle.zip results checkpoints figures environment.yml
from google.colab import files
files.download('/content/results_bundle.zip')